# Geo-FNO Dataset Sanity Checks

This notebook checks two things:
1. Whether samples have different net/background velocity statistics.
2. Whether plot color scaling (auto vs fixed) is fooling visual comparison.


In [ ]:
import os
import h5py
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.tri as tri

import torch

from geo_FNO_def import FNO2d, IPHI, get_global_L_from_h5

plt.rcParams['figure.figsize'] = (7, 4)
plt.rcParams['axes.grid'] = True


In [ ]:
# ----------------------------
# Config
# ----------------------------
H5_PATH = "/scratch/mnhagen/datasets/incompressible_euler/test.h5"
t_in = 0
t_out = -1
max_samples = None   # None = all

# Optional: enable model-based check if you want GT vs Pred with fixed scales
RUN_PRED_CHECK = False
FNO_CKPT  = "/scratch/mnhagen/models/geofno/cylinder_vel_t0_t-1_fno.pt"
IPHI_CKPT = "/scratch/mnhagen/models/geofno/cylinder_vel_t0_t-1_iphi.pt"

# Must match training if RUN_PRED_CHECK=True
modes = 20
width = 32
s1 = 80
s2 = 40
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


In [ ]:
# ----------------------------
# Helpers
# ----------------------------
def sample_stats(u):
    # u: (N,2)
    mean_vec = u.mean(axis=0)
    speed = np.linalg.norm(u, axis=1)
    return {
        "mean_ux": float(mean_vec[0]),
        "mean_uy": float(mean_vec[1]),
        "mean_speed": float(speed.mean()),
        "rms_speed": float(np.sqrt((speed ** 2).mean())),
        "p95_speed": float(np.quantile(speed, 0.95)),
        "max_speed": float(speed.max())
    }

def tri_plot(ax, pos, val, title, vmin=None, vmax=None, cmap="viridis"):
    triang = tri.Triangulation(pos[:, 0], pos[:, 1])
    tpc = ax.tripcolor(triang, val, shading="gouraud", cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_aspect("equal")
    ax.set_title(title)
    return tpc


In [ ]:
# ----------------------------
# Compute per-sample velocity stats
# ----------------------------
rows = []

with h5py.File(H5_PATH, "r") as f:
    keys = sorted([k for k in f.keys() if k.startswith("sample_")])
    if max_samples is not None:
        keys = keys[:max_samples]

    for k in keys:
        vel = f[k]["vel"][:]   # (T,N,2)
        u0 = vel[t_in].astype(np.float64)
        ut = vel[t_out].astype(np.float64)

        s0 = sample_stats(u0)
        st = sample_stats(ut)

        row = {
            "key": k,
            "mean_ux_in": s0["mean_ux"],
            "mean_uy_in": s0["mean_uy"],
            "mean_speed_in": s0["mean_speed"],
            "rms_speed_in": s0["rms_speed"],
            "mean_ux_out": st["mean_ux"],
            "mean_uy_out": st["mean_uy"],
            "mean_speed_out": st["mean_speed"],
            "rms_speed_out": st["rms_speed"],
            "d_mean_ux": st["mean_ux"] - s0["mean_ux"],
            "d_mean_uy": st["mean_uy"] - s0["mean_uy"],
            "d_rms_speed": st["rms_speed"] - s0["rms_speed"],
            "max_speed_out": st["max_speed"],
            "p95_speed_out": st["p95_speed"],
        }
        rows.append(row)

print(f"Samples processed: {len(rows)}")

for col in ["mean_ux_out", "mean_uy_out", "rms_speed_out", "d_mean_ux", "d_mean_uy", "d_rms_speed"]:
    vals = np.array([r[col] for r in rows])
    print(f"{col:>12s} | min={vals.min(): .4e}, max={vals.max(): .4e}, std={vals.std(): .4e}")


In [ ]:
# ----------------------------
# Histogram checks (net velocity / flow scale variability)
# ----------------------------
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

axes[0, 0].hist([r["mean_ux_out"] for r in rows], bins=40)
axes[0, 0].set_title("mean_ux_out")

axes[0, 1].hist([r["mean_uy_out"] for r in rows], bins=40)
axes[0, 1].set_title("mean_uy_out")

axes[0, 2].hist([r["rms_speed_out"] for r in rows], bins=40)
axes[0, 2].set_title("rms_speed_out")

axes[1, 0].hist([r["d_mean_ux"] for r in rows], bins=40)
axes[1, 0].set_title("d_mean_ux (out - in)")

axes[1, 1].hist([r["d_mean_uy"] for r in rows], bins=40)
axes[1, 1].set_title("d_mean_uy (out - in)")

axes[1, 2].hist([r["d_rms_speed"] for r in rows], bins=40)
axes[1, 2].set_title("d_rms_speed (out - in)")

plt.tight_layout()
plt.show()

In [ ]:
# ----------------------------
# Color-scale illusion demo on GT only (auto vs fixed)
# ----------------------------
idx_lo = int(np.argmin([r["rms_speed_out"] for r in rows]))
idx_hi = int(np.argmax([r["rms_speed_out"] for r in rows]))
k_lo = rows[idx_lo]["key"]
k_hi = rows[idx_hi]["key"]

with h5py.File(H5_PATH, "r") as f:
    pos_lo = f[k_lo]["pos"][:]
    pos_hi = f[k_hi]["pos"][:]
    u_lo = f[k_lo]["vel"][t_out].astype(np.float64)
    u_hi = f[k_hi]["vel"][t_out].astype(np.float64)

speed_lo = np.linalg.norm(u_lo, axis=1)
speed_hi = np.linalg.norm(u_hi, axis=1)
vmin = min(speed_lo.min(), speed_hi.min())
vmax = max(speed_lo.max(), speed_hi.max())

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
t0 = tri_plot(axes[0, 0], pos_lo, speed_lo, f"{k_lo} speed (auto)")
fig.colorbar(t0, ax=axes[0, 0])
t1 = tri_plot(axes[0, 1], pos_hi, speed_hi, f"{k_hi} speed (auto)")
fig.colorbar(t1, ax=axes[0, 1])
t2 = tri_plot(axes[1, 0], pos_lo, speed_lo, f"{k_lo} speed (fixed)", vmin=vmin, vmax=vmax)
fig.colorbar(t2, ax=axes[1, 0])
t3 = tri_plot(axes[1, 1], pos_hi, speed_hi, f"{k_hi} speed (fixed)", vmin=vmin, vmax=vmax)
fig.colorbar(t3, ax=axes[1, 1])

plt.tight_layout()
plt.show()

print("Lowest-rms sample:", k_lo, rows[idx_lo]["rms_speed_out"])
print("Highest-rms sample:", k_hi, rows[idx_hi]["rms_speed_out"])


In [ ]:
# ----------------------------
# Optional model check: GT vs Pred net velocity + fixed-scale plots
# ----------------------------
if RUN_PRED_CHECK:
    assert os.path.exists(FNO_CKPT), f"Missing checkpoint: {FNO_CKPT}"
    assert os.path.exists(IPHI_CKPT), f"Missing checkpoint: {IPHI_CKPT}"

    def fit_circle_kasa(x, y):
        A = np.stack([x, y, np.ones_like(x)], axis=1)
        b = -(x**2 + y**2)
        sol, *_ = np.linalg.lstsq(A, b, rcond=None)
        a, b_, c = sol
        xc = -a / 2.0
        yc = -b_ / 2.0
        r2 = (a*a + b_*b_) / 4.0 - c
        return float(xc), float(yc), float(np.sqrt(max(r2, 0.0)))

    def estimate_cylinder_from_label6(pos, node_type, boundary_label=6, band_frac=0.06):
        xy = pos[node_type == boundary_label]
        y = xy[:, 1]
        ymin, ymax = float(y.min()), float(y.max())
        band = band_frac * (ymax - ymin)
        xy_cyl = xy[(y > ymin + band) & (y < ymax - band)]
        return fit_circle_kasa(xy_cyl[:, 0], xy_cyl[:, 1])

    L_global, _ = get_global_L_from_h5(H5_PATH)
    model = FNO2d(modes, modes, width, in_channels=2, out_channels=2, is_mesh=False, s1=s1, s2=s2, L=L_global).to(device)
    model_iphi = IPHI(width=32, device=str(device)).to(device)
    model.load_state_dict(torch.load(FNO_CKPT, map_location=device), strict=False)
    model_iphi.load_state_dict(torch.load(IPHI_CKPT, map_location=device), strict=False)
    model.eval()
    model_iphi.eval()

    pred_rows = []
    with h5py.File(H5_PATH, "r") as f:
        keys = [r["key"] for r in rows]
        for k in keys:
            pos = f[k]["pos"][:].astype(np.float32)
            node_type = f[k]["node_type"][:]
            vel = f[k]["vel"][:]

            u_in = vel[t_in].astype(np.float32)
            u_gt = vel[t_out].astype(np.float32)
            xc, yc, rr = estimate_cylinder_from_label6(pos, node_type)

            code = np.zeros((42,), dtype=np.float32)
            code[0], code[1], code[2] = xc, yc, rr

            pos_t = torch.from_numpy(pos).unsqueeze(0).to(device)
            uin_t = torch.from_numpy(u_in).unsqueeze(0).to(device)
            code_t = torch.from_numpy(code).unsqueeze(0).to(device)

            with torch.no_grad():
                u_pred = model(uin_t, code=code_t, x_in=pos_t, x_out=pos_t, iphi=model_iphi).squeeze(0).cpu().numpy()

            gt_mean = u_gt.mean(axis=0)
            pred_mean = u_pred.mean(axis=0)
            pred_rows.append((k, gt_mean[0], gt_mean[1], pred_mean[0], pred_mean[1]))

    gt_ux = np.array([r[1] for r in pred_rows])
    gt_uy = np.array([r[2] for r in pred_rows])
    pr_ux = np.array([r[3] for r in pred_rows])
    pr_uy = np.array([r[4] for r in pred_rows])

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].scatter(gt_ux, pr_ux, s=10)
    lo, hi = min(gt_ux.min(), pr_ux.min()), max(gt_ux.max(), pr_ux.max())
    axes[0].plot([lo, hi], [lo, hi], 'k--', lw=1)
    axes[0].set_title("Mean u_x: GT vs Pred")
    axes[0].set_xlabel("GT")
    axes[0].set_ylabel("Pred")

    axes[1].scatter(gt_uy, pr_uy, s=10)
    lo, hi = min(gt_uy.min(), pr_uy.min()), max(gt_uy.max(), pr_uy.max())
    axes[1].plot([lo, hi], [lo, hi], 'k--', lw=1)
    axes[1].set_title("Mean u_y: GT vs Pred")
    axes[1].set_xlabel("GT")
    axes[1].set_ylabel("Pred")

    plt.tight_layout()
    plt.show()
else:
    print("RUN_PRED_CHECK=False, skipped model comparison.")
